# specs_novel — Phase 0: feasibility gate (Colab)

`specs_novel/roadmap.md` P0, thresholds pre-registered in `specs_novel/gate_report.md`.
**Generated by `build/mk_novel_p0.py` — do not hand-edit.** Nothing is trained here.

Runtime: **CPU, High-RAM** is enough (a GPU runtime with High-RAM also works). Roughly 1.5–2.5 h:
the DreaMS venv build, two streaming passes over `train.parquet`, and the multiprocessing stages.

| stage | question | gate |
|---|---|---|
| P0.2 | does replaying `casmi_dreams_colab.ipynb` cell 7 reproduce `corpus_meta.parquet` row by row, so CE can be attached to the cached embeddings? | G0.2 |
| P0.1a | does the vendored torch-only DreaMS (`dreams_infer.py`) match the official package and the cached embeddings? | G0.1 (Colab legs) |
| P0.3 | do the corpus structures survive stereo strip → SELFIES → SMILES → ik14? `MAX_LEN`, vocabulary | G0.3 |
| P0.4 | is the fast tracker exact, and are the masks sound on every true structure? | G0.4 |
| P0.5 | do the massspecgym pieces import, and which corpus structures are outside MassSpecGym? | G0.5 |

The official DreaMS package runs in its **own** venv (Python 3.11, its upstream pins) through
`dreams_official_worker.py`; this kernel never imports it, so RDKit 2026.03.3 here is undisturbed.
Order: P0.2 runs first because P0.1a samples its parity spectra from the replayed corpus.

Outputs go to Drive `casmi_2026/_novel/p0/`. The last cell writes `phase0_colab_gate.json` and
`kaggle_upload/` — upload the latter as the Kaggle dataset **casmi26-novel-offline**, then run
`casmi_novel_p0_kaggle.ipynb` (T4 ×2, internet off). Paste the final printout back into the chat.

In [ ]:
import os, sys, json, time, glob, shutil, hashlib, subprocess, importlib, re
import multiprocessing as mp
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/casmi_2026")
COMP = ROOT / "enveda-CASMI26-molecule-id-mass-spectra"
TRAIN = COMP / "train.parquet"
DOUT = ROOT / "_dreams_out"                     # casmi_dreams_colab.ipynb's WORK: corpus_meta, emb/, ckpt
RDKIT_WHL = next(iter(sorted((ROOT / "casmi26-offline-rdkit-2026033").glob("rdkit-*.whl"))), None)
GATE = ROOT / "_novel" / "p0"
UPLOAD = GATE / "kaggle_upload"                 # becomes the Kaggle dataset casmi26-novel-offline
WORK = Path("/content/novel_p0")
SRC = WORK / "src"
for p in (GATE, WORK, SRC):
    p.mkdir(parents=True, exist_ok=True)

DREAMS_COMMIT = "dbec3a0b514a99e5056cfccde4559fda8cfe8129"     # the commit dreams_infer.py was vendored from
MSGYM_COMMIT = "f259fe3780d5bd227fc6ece36ce6f397c2eef716"
SELFIES_PIN = "2.1.2"                           # MassSpecGym's setup.py pin
RDKIT_PIN = "2026.03.3"                         # the metric's RDKit
KAGGLE_USER = "YOUR_KAGGLE_USERNAME"            # only used in the dataset-metadata.json that is written

# the corpus exactly as casmi_dreams_colab.ipynb cell 1 CFG built it
TEST_ADDUCTS = ["[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",
                "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-"]
CAP_ENVEDA180, CAP_PER_LIB = 150_000, 400_000
CHUNK = 200_000                                 # rows per emb_*.npy

# gate thresholds -- specs_novel/gate_report.md, committed before this run
COS_OFFICIAL = 0.9999          # G0.1: official (a) vs vendored (b), every parity row
COS_CACHE = 0.999              # G0.1: vendored (b) vs cache (c), every parity row
COS_KAGGLE = 0.999             # G0.1: Kaggle (b) vs Colab (b) -- judged by the Kaggle notebook
RT_MIN = 0.99                  # G0.3: round-trip share
MAXLEN_Q = 0.999               # G0.3: MAX_LEN at p99.9 of the token lengths
N_PARITY = 200
N_TRACKER_PARITY = 10_000
N_SMOKE = 200                  # tracker walk repeated on Kaggle (recorded, not gating)
SEED = 0
NCPU = os.cpu_count()

GATE_RESULT = {"started": time.strftime("%Y-%m-%d %H:%M"), "dreams_commit": DREAMS_COMMIT,
               "msgym_commit": MSGYM_COMMIT, "selfies_pin": SELFIES_PIN, "vcpu": NCPU}

def save_gate():
    (GATE / "phase0_colab_gate.json").write_text(json.dumps(GATE_RESULT, indent=1, default=str))

def sh(cmd, timeout=None, env=None, check=True):
    print("$", cmd if len(cmd) < 300 else cmd[:300] + " ...", flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout, env=env)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode:
        print(r.stderr[-4000:])
        if check:
            raise RuntimeError(f"command failed ({r.returncode})")
    return r.stdout

def pool():
    return mp.get_context("fork").Pool(NCPU)

def sha1_text(s):
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

for name, p in [("train.parquet", TRAIN), ("corpus_meta.parquet", DOUT / "corpus_meta.parquet"),
                ("emb/", DOUT / "emb"), ("embedding_model.ckpt", DOUT / "embedding_model.ckpt"),
                ("rdkit wheel", RDKIT_WHL)]:
    print(f"{'ok ' if p is not None and p.exists() else '!! '} {name:22} {p}")
assert TRAIN.exists() and (DOUT / "corpus_meta.parquet").exists() and (DOUT / "emb").exists()
print(f"vCPU {NCPU}")

## Kernel packages: RDKit 2026.03.3 and the pinned selfies

In [ ]:
def kernel_version(mod):
    r = subprocess.run([sys.executable, "-c", f"import {mod}; print({mod}.__version__)"],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None

def v_match(v, target):
    if v is None:
        return False
    return v == target or v.replace(".0", ".") == target.replace(".0", ".")

if not v_match(kernel_version("rdkit"), RDKIT_PIN):
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    if RDKIT_WHL is not None and py_tag in RDKIT_WHL.name:
        sh(f"{sys.executable} -m pip install -q --no-deps {RDKIT_WHL}", check=False)
    if not v_match(kernel_version("rdkit"), RDKIT_PIN):
        print(f"[INFO] Offline wheel for {py_tag} not found or incompatible; installing from PyPI...", flush=True)
        sh(f"{sys.executable} -m pip install -q rdkit==2026.3.3")
def dist_version(dist):
    # selfies 2.1.2 still reports __version__ == "2.1.1"; the installed distribution's metadata is right
    r = subprocess.run([sys.executable, "-c", f"import importlib.metadata as m; print(m.version('{dist}'))"],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None

if dist_version("selfies") != SELFIES_PIN:
    sh(f"{sys.executable} -m pip install -q selfies=={SELFIES_PIN}")
import rdkit, selfies as sf, torch
assert v_match(rdkit.__version__, RDKIT_PIN), rdkit.__version__
assert dist_version("selfies") == SELFIES_PIN, dist_version("selfies")
GATE_RESULT["env"] = dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__,
                          pyarrow=pa.__version__, torch=torch.__version__, rdkit=rdkit.__version__,
                          selfies=dist_version("selfies"))
print(GATE_RESULT["env"])

## Project sources (embedded verbatim from `build/`)

Written to files so they can be imported here, by the multiprocessing workers and by the DreaMS
venv. Their SHA1s go into the Kaggle manifest; the Kaggle notebook embeds the same sources and
checks them against it.

In [ ]:
DREAMS_INFER_SRC = '"""DreaMS embedding inference, vendored: torch + numpy only (specs_novel plan §1.3 A2, roadmap P0.1).\n\nThe upstream package (pluskal-lab/DreaMS @ dbec3a0, local `DreamS/`) cannot be imported on Kaggle:\nits pins have no Python 3.12 wheels, `dreams.api` pulls in wandb / pyopenms / umap / igraph, and its\ncheckpoints pickle `dreams.*` classes. So the network code needed for the forward pass is copied here\nwith the SAME module and attribute names, which makes the upstream `state_dict` keys load `strict`.\n\n    DreamS/dreams/models/layers/fourier_features.py   FourierFeatures            (verbatim)\n    DreamS/dreams/models/layers/feed_forward.py       FeedForward                (verbatim; interpolate_interval inlined)\n    DreamS/dreams/models/dreams/layers.py             MultiheadAttention, TFeedForward (upstream\n                                                      name FeedForward), ScaleNorm, TransformerEncoder\n    DreamS/dreams/models/dreams/dreams.py             DreaMSBackbone: DreaMS.__init__ / forward /\n                                                      __normalize_spec, inference branches only\n    DreamS/dreams/models/heads/heads.py               ContrastiveHead: head(backbone(spec)[:, 0])\n\nThe weights come from `dreams_embed.pt`, written once on Colab by the P0.1a conversion (the real\npackage loads `embedding_model.ckpt` + `ssl_model.ckpt`; `args_to_dict` + `state_dict` are saved).\nFile layout: {"args_json": str, "state_dict": {name: tensor}, "meta_json": str}; plain types only, so\n`torch.load(weights_only=True)` works.\n\nPreprocessing is split in two, exactly as the cached corpus was built:\n    dreams_peaks     casmi_dreams_colab.ipynb cell 19 `_dreams_peaks` (= cell 7 `peak_list`): +2 Da cut,\n                     0.2 % floor, top-100, m/z ascending, intensity / max   -> (2, n) float32\n    to_model_input   upstream SpectrumPreprocessor(DataFormatA, n_highest_peaks=100) as used by\n                     `dreams_embeddings`: trim, zero-pad to 100, relative intensity, prepend\n                     [precursor_mz, 1.1], float32                           -> (101, 2) float32\nm/z above 1000 is NOT clipped (upstream never sets normalize_mzs); it is only outside pretraining.\n"""\nimport hashlib\nimport json\nfrom math import ceil\nfrom types import SimpleNamespace\nfrom typing import Sequence\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom torch.nn import Parameter\n\nN_PEAKS = 100             # checkpoint\'s n_highest_peaks (dreams/api.py `from_name`)\nPREC_INTENS = 1.1         # SpectrumPreprocessor default\nINT_FLOOR = 0.002         # casmi_dreams_colab.ipynb CFG\nDROP_ABOVE_PRECURSOR = 2.0\nMAX_MZ = 1000.0           # dreams/utils/dformats.py DataFormatA.max_mz\nMAX_TBXIC_STDEV = 1e-4    # DataFormatA.max_tbxic_stdev\nTRAINING_ONLY_PREFIXES = ("backbone.ff_out", "backbone.ff_out_intens", "backbone.mz_masking_loss",\n                          "backbone.ro_out")\n\n# Backbone args the inference path reads (dreams.py __init__, layers.py). Anything else is ignored.\nARG_KEYS = ("n_layers", "n_heads", "d_fourier", "d_peak", "d_mz_token", "charge_feature",\n            "fourier_strategy", "fourier_num_freqs", "fourier_min_freq", "fourier_trainable",\n            "dropout", "ff_fourier_depth", "ff_fourier_d", "no_ffs_bias", "ff_peak_depth",\n            "vanilla_transformer", "graphormer_mz_diffs", "graphormer_parametrized", "att_dropout",\n            "no_transformer_bias", "attn_mech", "ff_dropout", "residual_dropout", "pre_norm", "scnorm",\n            "dformat_max_mz", "dformat_max_tbxic_stdev")\n\n\ndef sha1(path, chunk=1 << 20):\n    h = hashlib.sha1()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(chunk), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\n# --------------------------------------------------------------------------------------------- #\n# Preprocessing\n# --------------------------------------------------------------------------------------------- #\n\ndef dreams_peaks(mz, it, prec, floor=INT_FLOOR, n=N_PEAKS, above=DROP_ABOVE_PRECURSOR):\n    mz = np.asarray(mz, np.float64); it = np.asarray(it, np.float64)\n    if len(mz) == 0: return None\n    k = mz <= prec + above; mz, it = mz[k], it[k]\n    if len(mz) == 0: return None\n    mx = it.max()\n    if not np.isfinite(mx) or mx <= 0: return None\n    k = it >= floor*mx; mz, it = mz[k], it[k]\n    if len(mz) < 2: return None\n    # Deterministic where cell 7 was not: its np.argsort (unstable) let the numpy build / CPU decide which\n    # tied peaks survive the top-n cut (G0.1-R). Ties at the cut go to the higher m/z; order is by m/z,\n    # then intensity, so the output does not depend on the input order either.\n    if len(mz) > n:\n        keep = np.lexsort((mz, it))[-n:]; mz, it = mz[keep], it[keep]\n    o = np.lexsort((it, mz)); mz, it = mz[o], it[o]\n    return np.stack([mz, it/it.max()]).astype(np.float32)   # (2, n)\n\n\ndef to_model_input(peaks, prec_mz, n=N_PEAKS, prec_intens=PREC_INTENS):\n    """(2, k) peak list -> (n + 1, 2) float32: upstream SpectrumPreprocessor.__call__ with its defaults."""\n    pl = np.array(peaks, copy=True)                                  # (2, k)\n    keep = np.sort(np.lexsort((pl[0], pl[1]))[-n:])                  # su.trim_peak_list, ties -> higher m/z\n    pl = pl[:, keep]\n    if pl.shape[1] < n:                                              # su.pad_peak_list\n        pl = np.pad(pl, ((0, 0), (0, n - pl.shape[1])), mode="constant", constant_values=0)\n    pl = np.array([pl[0], pl[1] / pl[1].max()])                      # su.to_rel_intensity\n    spec = np.vstack([np.array([prec_mz, prec_intens]), pl.T])       # su.prepend_precursor_peak\n    return spec.astype(np.float32, copy=False)\n\n\n# --------------------------------------------------------------------------------------------- #\n# Network (upstream code; see the module docstring for the file of origin)\n# --------------------------------------------------------------------------------------------- #\n\ndef interpolate_interval(a, b, n, only_inter=False, rounded=False):\n    x_min, x_max = min(a, b), max(a, b)\n    res = [x_min + i * (x_max - x_min) / (n + 1) for i in range(1 if only_inter else 0, n + 1 if only_inter else n + 2)]\n    if x_max == a:\n        res.reverse()\n    if rounded:\n        res = [round(x) for x in res]\n    return res\n\n\nclass FourierFeatures(nn.Module):\n    def __init__(self, strategy, x_min, x_max, trainable=True, funcs=\'both\', sigma=10, num_freqs=512):\n\n        assert strategy in {\'random\', \'voronov_et_al\', \'lin_float_int\'}\n        assert funcs in {\'both\', \'sin\', \'cos\'}\n        assert x_min < 1\n\n        super().__init__()\n        self.funcs = funcs\n        self.strategy = strategy\n        self.trainable = trainable\n        self.num_freqs = num_freqs\n\n        if strategy == \'random\':\n            self.b = torch.randn(num_freqs) * sigma\n        if self.strategy == \'voronov_et_al\':\n            self.b = torch.tensor(\n                [1 / (x_min * (x_max / x_min) ** (2 * i / (num_freqs - 2))) for i in range(1, num_freqs)],\n            )\n        elif self.strategy == \'lin_float_int\':\n            self.b = torch.tensor(\n                [1 / (x_min * i) for i in range(2, ceil(1 / x_min), 2)] +\n                [1 / (1 * i) for i in range(2, ceil(x_max), 1)],\n            )\n        self.b = self.b.unsqueeze(0)\n\n        self.b = nn.Parameter(self.b, requires_grad=self.trainable)\n        self.register_parameter(\'Fourier frequencies\', self.b)\n\n    def forward(self, x):\n        x = 2 * torch.pi * x @ self.b\n        if self.funcs == \'both\':\n            x = torch.cat((torch.cos(x), torch.sin(x)), dim=-1)\n        elif self.funcs == \'cos\':\n            x = torch.cos(x)\n        elif self.funcs == \'sin\':\n            x = torch.sin(x)\n        return x\n\n    def num_features(self):\n        return self.b.shape[1] if self.funcs != \'both\' else 2 * self.b.shape[1]\n\n\nclass FeedForward(nn.Module):\n\n    def __init__(self, in_dim, out_dim, hidden_dim, depth=None, act_last=True, act=nn.ReLU, bias=True, dropout=0):\n        super().__init__()\n\n        if isinstance(hidden_dim, int):\n            assert depth is not None\n            hidden_dim = [hidden_dim] * depth\n        elif hidden_dim == \'interpolated\':\n            assert depth is not None\n            hidden_dim = interpolate_interval(a=in_dim, b=out_dim, n=depth - 1, only_inter=True, rounded=True)\n        elif isinstance(hidden_dim, Sequence):  # e.g. is List or Tuple\n            depth = len(hidden_dim)\n        else:\n            raise ValueError\n\n        self.ff = nn.ModuleList([])\n        for l in range(depth):\n            d1 = hidden_dim[l - 1] if l != 0 else in_dim\n            d2 = hidden_dim[l] if l != depth - 1 else out_dim\n            self.ff.append(nn.Linear(d1, d2, bias=bias))\n            if l != depth - 1:\n                self.ff.append(nn.Dropout(p=dropout))\n            if l != depth - 1 or act_last:\n                self.ff.append(act())\n        self.ff = nn.Sequential(*self.ff)\n\n    def forward(self, x):\n        return self.ff(x)\n\n\nclass MultiheadAttention(nn.Module):\n    def __init__(self, args):\n        super(MultiheadAttention, self).__init__()\n        self.d_model = args.d_model\n        self.n_heads = args.n_heads\n        self.dropout = args.att_dropout\n        self.use_transformer_bias = not args.no_transformer_bias\n        self.attn_mech = args.attn_mech\n        self.d_graphormer_params = args.d_graphormer_params\n\n        if self.d_model % self.n_heads != 0:\n            raise ValueError(\'Required: d_model % n_heads == 0.\')\n\n        self.head_dim = self.d_model // self.n_heads\n        self.scale = self.head_dim ** -0.5\n\n        self.weights = Parameter(torch.Tensor(4 * self.d_model, self.d_model))\n        if self.use_transformer_bias:\n            self.biases = Parameter(torch.Tensor(4 * self.d_model))\n\n        if self.d_graphormer_params:\n            self.lin_graphormer = nn.Linear(self.d_graphormer_params, self.n_heads, bias=False)\n\n        mean = 0\n        std = (2 / (5 * self.d_model)) ** 0.5\n        nn.init.normal_(self.weights, mean=mean, std=std)\n        if self.use_transformer_bias:\n            nn.init.constant_(self.biases, 0.)\n\n        if self.attn_mech == \'additive_v\':\n            self.additive_v = Parameter(torch.Tensor(self.n_heads, self.head_dim))\n            nn.init.normal_(self.additive_v, mean=mean, std=std)\n\n    def forward(self, q, k, v, mask, graphormer_dists=None, do_proj_qkv=True):\n        bs, n, d = q.size()\n\n        def _split_heads(tensor):\n            bsz, length, d_model = tensor.size()\n            return tensor.reshape(bsz, length, self.n_heads, self.head_dim).transpose(1, 2)\n\n        if do_proj_qkv:\n            q, k, v = self.proj_qkv(q, k, v)\n\n        q = _split_heads(q)\n        k = _split_heads(k)\n        v = _split_heads(v)\n\n        if self.attn_mech == \'dot-product\':\n            att_weights = torch.einsum(\'bhnd,bhdm->bhnm\', q, k.transpose(-2, -1))\n        elif self.attn_mech == \'additive_v\' or self.attn_mech == \'additive_fixed\':\n            att_weights = (q.unsqueeze(-2) - k.unsqueeze(-3))\n            if self.attn_mech == \'additive_v\':\n                att_weights = (att_weights * self.additive_v.unsqueeze(0).unsqueeze(2).unsqueeze(3))\n            att_weights = att_weights.sum(dim=-1)\n        else:\n            raise NotImplementedError(f\'"{self.attn_mech}" attention mechanism is not implemented.\')\n        att_weights = att_weights * self.scale\n\n        if graphormer_dists is not None:\n            if self.d_graphormer_params:\n                att_bias = self.lin_graphormer(graphormer_dists).permute(0, 3, 1, 2)\n            else:\n                att_bias = graphormer_dists.sum(dim=-1).unsqueeze(1)\n            att_weights = att_weights + att_bias\n\n        if mask is not None:\n            att_weights.masked_fill_(mask.unsqueeze(1).unsqueeze(-1), -1e9)\n\n        att_weights = F.softmax(att_weights, dim=-1)\n        att_weights = F.dropout(att_weights, p=self.dropout, training=self.training)\n        _att_weights = att_weights.reshape(-1, n, n)\n        output = torch.bmm(_att_weights, v.reshape(bs * self.n_heads, -1, self.head_dim))\n        output = output.reshape(bs, self.n_heads, n, self.head_dim).transpose(1, 2).reshape(bs, n, -1)\n        output = self.proj_o(output)\n\n        return output, att_weights\n\n    def proj_qkv(self, q, k, v):\n        qkv_same = q.data_ptr() == k.data_ptr() == v.data_ptr()\n        kv_same = k.data_ptr() == v.data_ptr()\n\n        if qkv_same:\n            q, k, v = self._proj(q, end=3 * self.d_model).chunk(3, dim=-1)\n        elif kv_same:\n            q = self._proj(q, end=self.d_model)\n            k, v = self._proj(k, start=self.d_model, end=3 * self.d_model).chunk(2, dim=-1)\n        else:\n            q = self.proj_q(q)\n            k = self.proj_k(k)\n            v = self.proj_v(v)\n\n        return q, k, v\n\n    def _proj(self, x, start=0, end=None):\n        weight = self.weights[start:end, :]\n        bias = None if not self.use_transformer_bias else self.biases[start:end]\n        return F.linear(x, weight=weight, bias=bias)\n\n    def proj_q(self, q):\n        return self._proj(q, end=self.d_model)\n\n    def proj_k(self, k):\n        return self._proj(k, start=self.d_model, end=2 * self.d_model)\n\n    def proj_v(self, v):\n        return self._proj(v, start=2 * self.d_model, end=3 * self.d_model)\n\n    def proj_o(self, x):\n        return self._proj(x, start=3 * self.d_model)\n\n\nclass TFeedForward(nn.Module):\n    """Upstream `dreams.models.dreams.layers.FeedForward` (renamed: clashes with the peak FeedForward)."""\n    def __init__(self, args):\n        super(TFeedForward, self).__init__()\n        self.dropout = args.ff_dropout\n        self.d_model = args.d_model\n        self.ff_dim = 4 * args.d_model\n        self.use_transformer_bias = not args.no_transformer_bias\n\n        self.in_proj = nn.Linear(self.d_model, self.ff_dim, bias=self.use_transformer_bias)\n        self.out_proj = nn.Linear(self.ff_dim, self.d_model, bias=self.use_transformer_bias)\n\n        mean = 0\n        std = (2 / (self.ff_dim + self.d_model)) ** 0.5\n        nn.init.normal_(self.in_proj.weight, mean=mean, std=std)\n        nn.init.normal_(self.out_proj.weight, mean=mean, std=std)\n        if self.use_transformer_bias:\n            nn.init.constant_(self.in_proj.bias, 0.)\n            nn.init.constant_(self.out_proj.bias, 0.)\n\n    def forward(self, x):\n        y = F.relu(self.in_proj(x))\n        y = F.dropout(y, p=self.dropout, training=self.training)\n        return self.out_proj(y)\n\n\nclass ScaleNorm(nn.Module):\n    def __init__(self, scale, eps=1e-5):\n        super(ScaleNorm, self).__init__()\n        self.scale = Parameter(torch.tensor(scale))\n        self.eps = eps\n\n    def forward(self, x):\n        norm = self.scale / torch.norm(x, dim=-1, keepdim=True).clamp(min=self.eps)\n        return x * norm\n\n\nclass TransformerEncoder(nn.Module):\n    def __init__(self, args):\n        super(TransformerEncoder, self).__init__()\n        self.residual_dropout = args.residual_dropout\n        self.n_layers = args.n_layers\n        self.pre_norm = args.pre_norm\n\n        self.atts = nn.ModuleList([MultiheadAttention(args) for _ in range(self.n_layers)])\n        self.ffs = nn.ModuleList([TFeedForward(args) for _ in range(self.n_layers)])\n\n        num_scales = self.n_layers * 2 + 1 if self.pre_norm else self.n_layers * 2\n        if args.scnorm:\n            self.scales = nn.ModuleList([ScaleNorm(args.d_model ** 0.5) for _ in range(num_scales)])\n        else:\n            self.scales = nn.ModuleList([nn.LayerNorm(args.d_model) for _ in range(num_scales)])\n\n    def _layer_forward(self, i, x, src_mask, graphormer_dists):\n        pre_norm = self.pre_norm\n        post_norm = not pre_norm\n        att = self.atts[i]\n        ff = self.ffs[i]\n        att_scale = self.scales[2 * i]\n        ff_scale = self.scales[2 * i + 1]\n\n        residual = x\n        x = att_scale(x) if pre_norm else x\n        x, _ = att(q=x, k=x, v=x, mask=src_mask, graphormer_dists=graphormer_dists)\n        x = residual + F.dropout(x, p=self.residual_dropout, training=self.training)\n        x = att_scale(x) if post_norm else x\n\n        residual = x\n        x = ff_scale(x) if pre_norm else x\n        x = ff(x)\n        x = residual + F.dropout(x, p=self.residual_dropout, training=self.training)\n        x = ff_scale(x) if post_norm else x\n        return x\n\n    def forward(self, src_inputs, src_mask, graphormer_dists=None):\n        x = F.dropout(src_inputs, p=self.residual_dropout, training=self.training)\n        for i in range(self.n_layers):\n            x = self._layer_forward(i, x, src_mask, graphormer_dists)\n        x = self.scales[-1](x) if self.pre_norm else x\n        return x\n\n\nclass DreaMSBackbone(nn.Module):\n    """Upstream `DreaMS` (LightningModule) reduced to the modules the embedding forward pass uses."""\n\n    def __init__(self, args):\n        super().__init__()\n        if args.charge_feature:\n            raise NotImplementedError("charge_feature=True: the embedding checkpoint is used without charge")\n        if args.d_mz_token or not args.d_fourier:\n            raise NotImplementedError("only the Fourier-feature m/z encoding is vendored")\n        if args.vanilla_transformer:\n            raise NotImplementedError("only the DreaMS TransformerEncoder is vendored")\n        self.n_layers = args.n_layers\n        self.d_fourier = args.d_fourier\n        self.d_peak = args.d_peak\n        self.d_model = sum(d for d in [args.d_fourier, args.d_peak, args.d_mz_token] if d)\n        args.d_model = self.d_model\n        self.max_mz = args.dformat_max_mz\n        self.graphormer_mz_diffs = args.graphormer_mz_diffs\n        if args.graphormer_mz_diffs and args.graphormer_parametrized:\n            args.d_graphormer_params = args.d_fourier if args.d_fourier else 1\n        else:\n            args.d_graphormer_params = 0\n\n        token_dim = 2\n        self.fourier_enc = FourierFeatures(strategy=args.fourier_strategy, num_freqs=args.fourier_num_freqs,\n                                           x_min=args.dformat_max_tbxic_stdev if not args.fourier_min_freq else args.fourier_min_freq,\n                                           x_max=args.dformat_max_mz,\n                                           trainable=args.fourier_trainable)\n        self.ff_fourier = FeedForward(in_dim=self.fourier_enc.num_features(), out_dim=args.d_fourier, dropout=args.dropout,\n                                      depth=args.ff_fourier_depth, hidden_dim=args.ff_fourier_d, bias=not args.no_ffs_bias)\n        self.ff_peak = FeedForward(in_dim=token_dim, hidden_dim=args.d_peak, out_dim=args.d_peak, depth=args.ff_peak_depth,\n                                   dropout=args.dropout, bias=not args.no_ffs_bias)\n        self.transformer_encoder = TransformerEncoder(args)\n\n    def forward(self, spec):\n        padding_mask = spec[:, :, 0] == 0\n        peak_embs = self.ff_peak(spec / spec.new_tensor([self.max_mz, 1.]))\n        fourier_features = self.ff_fourier(self.fourier_enc(spec[..., [0]]))\n        spec = torch.cat([peak_embs, fourier_features], dim=-1)\n        graphormer_dists = None\n        if self.graphormer_mz_diffs:\n            graphormer_dists = fourier_features.unsqueeze(2) - fourier_features.unsqueeze(1)\n        return self.transformer_encoder(spec, padding_mask, graphormer_dists)\n\n\nclass DreamsEmbedder(nn.Module):\n    """Upstream ContrastiveHead at inference: Linear(d_model, d_model) on the precursor token."""\n\n    def __init__(self, args):\n        super().__init__()\n        self.backbone = DreaMSBackbone(args)\n        self.head = nn.Linear(self.backbone.d_model, self.backbone.d_model, bias=True)\n\n    def forward(self, spec):\n        return self.head(self.backbone(spec)[:, 0, ...])\n\n\n# --------------------------------------------------------------------------------------------- #\n# Conversion (Colab, with the real package loaded) and loading (anywhere)\n# --------------------------------------------------------------------------------------------- #\n\ndef args_to_dict(args):\n    """Upstream backbone `args` (argparse Namespace; `dformat` is a DataFormatA object) -> plain dict."""\n    src = vars(args) if not isinstance(args, dict) else args\n    out = {k: src[k] for k in ARG_KEYS if k in src}\n    dformat = src.get("dformat")\n    out["dformat_max_mz"] = float(getattr(dformat, "max_mz", MAX_MZ))\n    out["dformat_max_tbxic_stdev"] = float(getattr(dformat, "max_tbxic_stdev", MAX_TBXIC_STDEV))\n    missing = [k for k in ARG_KEYS if k not in out]\n    assert not missing, f"backbone args lack {missing}"\n    return out\n\n\ndef inference_state_dict(state_dict):\n    """Drop the training-only backbone heads that PreTrainedModel deletes after loading (api.py)."""\n    return {k: v for k, v in state_dict.items() if not k.startswith(TRAINING_ONLY_PREFIXES)}\n\n\ndef save_dreams(path, args, state_dict, meta):\n    torch.save({"args_json": json.dumps(args_to_dict(args), sort_keys=True),\n                "state_dict": {k: v.detach().cpu().float().contiguous() for k, v in\n                               inference_state_dict(state_dict).items()},\n                "meta_json": json.dumps(meta, sort_keys=True)}, path)\n\n\ndef build_embedder(args_dict):\n    return DreamsEmbedder(SimpleNamespace(**args_dict))\n\n\ndef load_dreams(path, device="cpu", expect_sha1=None):\n    """-> (DreamsEmbedder in eval / float32 on device, meta dict). strict load: every key must match."""\n    if expect_sha1 is not None:\n        got = sha1(path)\n        assert got == expect_sha1, f"{path}: sha1 {got} != expected {expect_sha1}"\n    blob = torch.load(path, map_location="cpu", weights_only=True)\n    model = build_embedder(json.loads(blob["args_json"]))\n    model.load_state_dict(blob["state_dict"], strict=True)\n    return model.to(device=device, dtype=torch.float32).eval(), json.loads(blob["meta_json"])\n\n\n@torch.inference_mode()\ndef dreams_embed(model, peak_lists, precursors, batch_size=64):\n    """peak_lists: (2, k) arrays from `dreams_peaks`; precursors: m/z. -> (n, d_model) float32 numpy."""\n    dev = next(model.parameters()).device\n    out = []\n    for lo in range(0, len(peak_lists), batch_size):\n        x = np.stack([to_model_input(p, m) for p, m in\n                      zip(peak_lists[lo:lo + batch_size], precursors[lo:lo + batch_size])])\n        out.append(model(torch.from_numpy(x).to(dev)).float().cpu().numpy())\n    d = model.head.out_features\n    return np.concatenate(out) if out else np.zeros((0, d), np.float32)\n'
NOVEL_DECODER_SRC = '"""specs_novel single source of truth for the SELFIES side (plan §4, §5.5). P0 part: no model yet.\n\n    SelfiesVocab        token <-> id, the same ids as MassSpecGym\'s SelfiesTokenizer (JSON, no `tokenizers`)\n    TokenTable          per-token grammar facts, precomputed with selfies\' own grammar_rules\n    Tracker             incremental SELFIES derivation state -> legal-token mask (plan §4.1-4.2)\n    reference_state     the slow truth: selfies.decoder(prefix) + RDKit counts (P0.4 parity)\n    walk_truth          mask soundness: every true next token legal, </s> legal at the end (P0.4)\n    prep_structure      stereo/isotope strip -> canonical SMILES -> SELFIES (A9, P0.3)\n\nThe Tracker replays `selfies.decoder._derive_mol_from_symbols` token by token. That function is\nrecursive (a branch derives a sub-chain of Q+1 symbols); here its call stack is an explicit list of\nframes [state, prev_atom, budget]. Every consumed symbol lowers the budget of every open frame, the\nway each caller adds its child\'s n_derived; a frame closes when its budget reaches 0. A frame whose\nstate becomes None (no free valence left) swallows the rest of its budget, and in the main chain that\nmeans every later symbol -- SELFIES silently skips those atoms (A7), so the mask never offers them.\n\nMasks (plan §4.2):\n    pending index symbols      only the 16 INDEX_ALPHABET symbols (the encoder emits nothing else there)\n    atom token, element e      frame active, the atom would be placed, and remaining[e] > 0\n    [Branch*]                  frame state > 1 (otherwise selfies ignores it)\n    [Ring*]                    frame state > 0\n    </s>                       no pending index and every heavy-atom count exactly used\n`unsat()` is rings + pi bonds of the molecule the prefix decodes to, with the ring bonds formed the way\n`_form_rings_bilocally` forms them at the end (orders capped by the free valence left). It feeds the\noptional DBE prune, which is only applied where P0.4 shows it sound (A8).\n\nNeeds numpy + selfies (pinned); RDKit only for reference_state / prep_structure / ik14.\n"""\nimport json\nimport re\n\nimport numpy as np\nimport selfies as sf\nfrom selfies.constants import INDEX_CODE\nfrom selfies.grammar_rules import (get_index_from_selfies, next_atom_state, next_branch_state,\n                                   next_ring_state, process_atom_symbol, process_branch_symbol,\n                                   process_ring_symbol)\n\nPAD, SOS, EOS, UNK = "<pad>", "<s>", "</s>", "<unk>"\nALLOWED_ELEMENTS = ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I")\nDBE_SAFE_ELEMENTS = frozenset(("C", "H", "N", "O", "F", "Cl", "Br", "I"))   # no S / P (plan §4.2)\nK_SPECIAL, K_ATOM, K_BRANCH, K_RING, K_OTHER = 0, 1, 2, 3, 4\n_FORMULA_RE = re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef check_constraints():\n    """The token table and the tracker assume selfies\' default semantic constraints."""\n    assert sf.get_semantic_constraints() == sf.get_preset_constraints("default"), \\\n        "selfies semantic constraints are not the default preset"\n\n\n# --------------------------------------------------------------------------------------------- #\n# Vocabulary\n# --------------------------------------------------------------------------------------------- #\n\ndef vocab_from_alphabet(alphabet):\n    """SelfiesTokenizer\'s id layout (massspecgym/models/tokenizers.py:79-85 + add_special_tokens):\n    sorted alphabet 0..n-1, <unk> n, then <pad> <s> </s> as the next added tokens."""\n    alphabet = sorted(set(alphabet) - {PAD, SOS, EOS, UNK})\n    tok2id = {s: i for i, s in enumerate(alphabet)}\n    tok2id[UNK] = len(tok2id)\n    for s in (PAD, SOS, EOS):\n        tok2id[s] = len(tok2id)\n    return tok2id\n\n\nclass SelfiesVocab:\n    def __init__(self, tok2id):\n        ids = sorted(tok2id.values())\n        assert ids == list(range(len(ids))), "token ids must be 0..n-1"\n        for s in (PAD, SOS, EOS, UNK):\n            assert s in tok2id, f"special token {s} missing"\n        self.tok2id = dict(tok2id)\n        self.id2tok = [None] * len(ids)\n        for s, i in self.tok2id.items():\n            self.id2tok[i] = s\n        self.pad_id, self.sos_id, self.eos_id, self.unk_id = (self.tok2id[s] for s in (PAD, SOS, EOS, UNK))\n\n    def __len__(self):\n        return len(self.id2tok)\n\n    @classmethod\n    def from_alphabet(cls, alphabet):\n        return cls(vocab_from_alphabet(alphabet))\n\n    @classmethod\n    def from_json(cls, path):\n        with open(path, encoding="utf-8") as f:\n            return cls(json.load(f))\n\n    def to_json(self, path):\n        with open(path, "w", encoding="utf-8") as f:\n            json.dump(self.tok2id, f, sort_keys=True, indent=0)\n\n    def encode(self, selfies_str):\n        """SELFIES string -> ids, no <s>/</s>."""\n        return [self.tok2id.get(t, self.unk_id) for t in sf.split_selfies(selfies_str)]\n\n    def decode(self, ids):\n        """ids -> SELFIES string; special tokens dropped."""\n        return "".join(self.id2tok[i] for i in ids if i not in (self.pad_id, self.sos_id, self.eos_id,\n                                                                self.unk_id))\n\n\n# --------------------------------------------------------------------------------------------- #\n# Token table\n# --------------------------------------------------------------------------------------------- #\n\nclass TokenTable:\n    """Per-token facts over a vocabulary, from selfies\' own grammar_rules (so they follow the pin)."""\n\n    def __init__(self, vocab):\n        check_constraints()\n        n = len(vocab)\n        self.vocab = vocab\n        self.kind = np.full(n, K_OTHER, np.int8)\n        self.elem = np.full(n, -1, np.int16)           # index into self.elements\n        self.bond = np.zeros(n, np.int8)               # requested bond order (atom / branch / ring type)\n        self.cap = np.zeros(n, np.int8)                # atom bonding capacity (after explicit H, charge)\n        self.nidx = np.zeros(n, np.int8)               # index symbols read after a branch / ring\n        self.is_index = np.zeros(n, bool)              # one of the 16 INDEX_ALPHABET symbols\n        self.index_code = np.zeros(n, np.int16)\n        elements = []\n        for i, t in enumerate(vocab.id2tok):\n            if t in (PAD, SOS, EOS, UNK):\n                self.kind[i] = K_SPECIAL\n                continue\n            if t in INDEX_CODE:\n                self.is_index[i] = True\n                self.index_code[i] = INDEX_CODE[t]\n            if "ch" == t[-4:-2]:                       # the same dispatch order as the decoder\n                out = process_branch_symbol(t)\n                if out is not None:\n                    self.kind[i], self.bond[i], self.nidx[i] = K_BRANCH, out[0], out[1]\n            elif "ng" == t[-4:-2]:\n                out = process_ring_symbol(t)\n                if out is not None:\n                    self.kind[i], self.bond[i], self.nidx[i] = K_RING, out[0], out[1]\n            elif "eps" in t:\n                pass\n            else:\n                out = process_atom_symbol(t)\n                if out is not None:\n                    (bond, _), atom = out\n                    if atom.element not in elements:\n                        elements.append(atom.element)\n                    self.kind[i] = K_ATOM\n                    self.elem[i] = elements.index(atom.element)\n                    self.bond[i] = bond\n                    self.cap[i] = atom.bonding_capacity\n        self.elements = tuple(elements)\n        self.elem_ix = {e: j for j, e in enumerate(self.elements)}\n        self.h_ix = self.elem_ix.get("H", -1)\n        self.is_atom = self.kind == K_ATOM\n        self.is_branch = self.kind == K_BRANCH\n        self.is_ring = self.kind == K_RING\n        self.atom_cap_pos = self.is_atom & (self.cap > 0)\n        self.elem_safe = np.maximum(self.elem, 0)\n\n    def counts_vector(self, counts):\n        """{element: count} -> vector over self.elements, plus the count of elements the vocab lacks."""\n        v = np.zeros(len(self.elements), np.int32)\n        missing = 0\n        for e, c in counts.items():\n            if e in self.elem_ix:\n                v[self.elem_ix[e]] = c\n            elif e != "H":\n                missing += c\n        return v, missing\n\n\n# --------------------------------------------------------------------------------------------- #\n# Tracker\n# --------------------------------------------------------------------------------------------- #\n\nINF = float("inf")\n\n\nclass Tracker:\n    """Persistent SELFIES derivation state. `push` returns a new Tracker; the old one is unchanged.\n\n    formula: {element: count} incl. H, or None (then only counting, no count masks)."""\n\n    __slots__ = ("tt", "frames", "pending", "caps", "bc", "bonds", "rings", "counts", "target",\n                 "missing", "n_tokens", "broken")\n\n    def __init__(self, table, formula=None):\n        self.tt = table\n        self.frames = [[0, None, INF]]          # [state (int | None), prev atom index, budget]\n        self.pending = None                     # [kind, n_left, symbols, payload]\n        self.caps = []                          # bonding capacity per placed atom\n        self.bc = []                            # bond order sum per atom (chain bonds only)\n        self.bonds = {}                         # (a, b) a < b -> order (chain bonds only)\n        self.rings = []                         # (left atom, right atom, order), formed at the end\n        self.counts = np.zeros(len(table.elements), np.int32)\n        if formula is None:\n            self.target, self.missing = None, 0\n        else:\n            self.target, self.missing = table.counts_vector(formula)\n        self.n_tokens = 0\n        self.broken = False\n\n    def _copy(self):\n        t = Tracker.__new__(Tracker)\n        t.tt = self.tt\n        t.frames = [f[:] for f in self.frames]\n        t.pending = None if self.pending is None else [self.pending[0], self.pending[1],\n                                                       self.pending[2][:], self.pending[3]]\n        t.caps, t.bc, t.bonds, t.rings = self.caps[:], self.bc[:], dict(self.bonds), self.rings[:]\n        t.counts = self.counts.copy()\n        t.target, t.missing, t.n_tokens, t.broken = self.target, self.missing, self.n_tokens, self.broken\n        return t\n\n    # -- derivation -------------------------------------------------------------------------- #\n\n    def _consume(self):\n        for f in self.frames:\n            f[2] -= 1\n\n    def _close_exhausted(self):\n        while len(self.frames) > 1 and self.frames[-1][2] <= 0:\n            self.frames.pop()\n\n    def _add_atom(self, cap):\n        self.caps.append(int(cap))\n        self.bc.append(0)\n        return len(self.caps) - 1\n\n    def push(self, tok_id):\n        t = self._copy()\n        t._apply(int(tok_id))\n        return t\n\n    def _apply(self, i):\n        tt = self.tt\n        self.n_tokens += 1\n        self._consume()\n        if self.pending is not None:                      # an index symbol of a branch / ring\n            p = self.pending\n            p[2].append(tt.vocab.id2tok[i])\n            p[1] -= 1\n            if p[1] == 0:\n                self.pending = None\n                q = get_index_from_selfies(*p[2])\n                f = self.frames[-1]\n                if p[0] == K_BRANCH:\n                    self.frames.append([p[3], f[1], q + 1])\n                else:\n                    lidx = max(0, f[1] - (q + 1))\n                    self.rings.append((lidx, f[1], p[3]))\n            self._close_exhausted()\n            return\n        f = self.frames[-1]\n        state = f[0]\n        kind = tt.kind[i]\n        if state is None or kind in (K_SPECIAL, K_OTHER):\n            if kind == K_OTHER and state is not None and "eps" in tt.vocab.id2tok[i]:\n                f[0] = 0 if state == 0 else None\n            elif kind == K_SPECIAL:\n                self.broken = True                        # specials never occur inside a SELFIES string\n        elif kind == K_BRANCH:\n            if state > 1:\n                binit, f[0] = next_branch_state(int(tt.bond[i]), state)\n                self.pending = [K_BRANCH, int(tt.nidx[i]), [], binit]\n        elif kind == K_RING:\n            if state != 0:\n                order, f[0] = next_ring_state(int(tt.bond[i]), state)\n                self.pending = [K_RING, int(tt.nidx[i]), [], order]\n        else:                                             # atom\n            cap = int(tt.cap[i])\n            order, nxt = next_atom_state(int(tt.bond[i]), cap, state)\n            if order == 0:\n                if state == 0:\n                    f[1] = self._add_atom(cap)\n                    self.counts[tt.elem[i]] += 1\n                else:\n                    self.broken = True                    # selfies would bond from an unplaced atom\n            else:\n                a = self._add_atom(cap)\n                self.counts[tt.elem[i]] += 1\n                src = f[1]\n                self.bonds[(src, a)] = order\n                self.bc[src] += order\n                self.bc[a] += order\n                f[1] = a\n            f[0] = nxt\n        self._close_exhausted()\n\n    # -- queries ----------------------------------------------------------------------------- #\n\n    def remaining(self):\n        return None if self.target is None else self.target - self.counts\n\n    def heavy_done(self):\n        r = self.remaining()\n        if r is None:\n            return True\n        r = r.copy()\n        if self.tt.h_ix >= 0:\n            r[self.tt.h_ix] = 0\n        return self.missing == 0 and not r.any()\n\n    def eos_legal(self):\n        return (not self.broken and self.pending is None and len(self.caps) > 0 and self.heavy_done())\n\n    def legal_mask(self):\n        """numpy bool over the vocabulary."""\n        tt = self.tt\n        n = len(tt.kind)\n        if self.broken:\n            return np.zeros(n, bool)\n        if self.pending is not None:\n            return tt.is_index.copy()\n        state = self.frames[-1][0]\n        mask = np.zeros(n, bool)\n        if state is not None:\n            placed = tt.is_atom if state == 0 else tt.atom_cap_pos\n            if self.target is not None:\n                placed = placed & (self.remaining()[tt.elem_safe] > 0)\n            mask |= placed\n            if state > 1:\n                mask |= tt.is_branch\n            if state > 0:\n                mask |= tt.is_ring\n        if self.eos_legal():\n            mask[tt.vocab.eos_id] = True\n        return mask\n\n    def unsat(self):\n        """Rings + pi bonds of the molecule this prefix decodes to (selfies\' _form_rings_bilocally)."""\n        n = len(self.caps)\n        if n == 0:\n            return 0\n        bc, bonds = self.bc[:], dict(self.bonds)\n        for l, r, order in self.rings:\n            if l == r:\n                continue\n            lfree, rfree = self.caps[l] - bc[l], self.caps[r] - bc[r]\n            if lfree <= 0 or rfree <= 0:\n                continue\n            order = min(order, lfree, rfree)\n            key = (min(l, r), max(l, r))\n            if key in bonds:\n                new = min(order + bonds[key], 3)\n                bc[l] += new - bonds[key]\n                bc[r] += new - bonds[key]\n                bonds[key] = new\n            else:\n                bonds[key] = order\n                bc[l] += order\n                bc[r] += order\n        return sum(bonds.values()) - n + 1\n\n    def element_counts(self):\n        return {e: int(c) for e, c in zip(self.tt.elements, self.counts) if c}\n\n\n# --------------------------------------------------------------------------------------------- #\n# Formulae\n# --------------------------------------------------------------------------------------------- #\n\ndef parse_formula(formula):\n    counts = {}\n    for e, c in _FORMULA_RE.findall(formula):\n        counts[e] = counts.get(e, 0) + (int(c) if c else 1)\n    return counts\n\n\ndef dbe(counts):\n    """Rings + pi bonds implied by a formula with standard valences (C4 N3 O2 X1 H1)."""\n    x = sum(counts.get(e, 0) for e in ("F", "Cl", "Br", "I"))\n    return 1 + counts.get("C", 0) - (counts.get("H", 0) + x) / 2 + counts.get("N", 0) / 2\n\n\ndef dbe_prune_applies(counts):\n    return set(e for e, c in counts.items() if c) <= DBE_SAFE_ELEMENTS\n\n\n# --------------------------------------------------------------------------------------------- #\n# RDKit side: structure prep, ik14, reference tracker\n# --------------------------------------------------------------------------------------------- #\n\ndef _chem():\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog("rdApp.*")\n    return Chem\n\n\ndef mol_formula(mol):\n    """{element: count} incl. total H (implicit + explicit) of an RDKit mol."""\n    counts = {}\n    for a in mol.GetAtoms():\n        s = a.GetSymbol()\n        counts[s] = counts.get(s, 0) + 1\n        h = a.GetTotalNumHs()\n        if h:\n            counts["H"] = counts.get("H", 0) + h\n    return counts\n\n\ndef ik14(smiles):\n    Chem = _chem()\n    m = Chem.MolFromSmiles(smiles) if smiles else None\n    if m is None:\n        return None\n    k = Chem.MolToInchiKey(m)\n    return k[:14] if k else None\n\n\ndef prep_structure(smiles, elements=ALLOWED_ELEMENTS):\n    """-> (status, canonical_smiles, selfies, formula). status \'ok\' or the reason it is excluded:\n    \'parse\', \'multi\' (more than one component), \'element\', \'encode\' (selfies constraints)."""\n    Chem = _chem()\n    m = Chem.MolFromSmiles(smiles) if smiles else None\n    if m is None:\n        return "parse", None, None, None\n    if len(Chem.GetMolFrags(m)) > 1:\n        return "multi", None, None, None\n    if any(a.GetSymbol() not in elements for a in m.GetAtoms()):\n        return "element", None, None, None\n    Chem.RemoveStereochemistry(m)\n    for a in m.GetAtoms():\n        a.SetIsotope(0)\n    can = Chem.MolToSmiles(m)\n    m = Chem.MolFromSmiles(can)\n    try:\n        s = sf.encoder(can, strict=True)\n    except Exception:\n        return "encode", can, None, None\n    if s is None:\n        return "encode", can, None, None\n    return "ok", can, s, mol_formula(m)\n\n\ndef reference_state(tokens):\n    """Slow truth for a token prefix: (element counts incl. explicit H atoms, rings + pi bonds)."""\n    Chem = _chem()\n    smi = sf.decoder("".join(tokens))\n    if not smi:\n        return {}, 0\n    m = Chem.MolFromSmiles(smi, sanitize=False)\n    counts = {}\n    for a in m.GetAtoms():\n        counts[a.GetSymbol()] = counts.get(a.GetSymbol(), 0) + 1\n    order = sum(int(b.GetBondTypeAsDouble()) for b in m.GetBonds())\n    n_comp = len(Chem.GetMolFrags(m, sanitizeFrags=False))\n    return counts, order - m.GetNumAtoms() + n_comp\n\n\ndef tracker_parity(table, tokens):\n    """Fast vs reference at every prefix. -> list of (prefix length, what, fast, reference)."""\n    vocab = table.vocab\n    t = Tracker(table)\n    bad = []\n    for k, tok in enumerate(tokens, 1):\n        t = t.push(vocab.tok2id[tok])\n        ref_counts, ref_u = reference_state(tokens[:k])\n        if t.element_counts() != ref_counts:\n            bad.append((k, "counts", t.element_counts(), ref_counts))\n        if t.pending is None and t.unsat() != ref_u:\n            bad.append((k, "unsat", t.unsat(), ref_u))\n        if t.broken:\n            bad.append((k, "broken", None, None))\n    return bad\n\n\ndef walk_truth(table, tokens, formula, max_len=None, dbe_prune=False):\n    """Mask soundness for one truth. -> list of (position, rule, token) violations.\n\n    rules: \'count\' (atom masked, element exhausted), \'grammar\' (masked for any other reason),\n    \'eos\' (</s> not legal after the last token), \'length\' (> max_len), \'dbe\' (unsat above DBE)."""\n    vocab = table.vocab\n    t = Tracker(table, formula)\n    limit = dbe(formula) if dbe_prune and dbe_prune_applies(formula) else None\n    bad = []\n    for k, tok in enumerate(tokens):\n        i = vocab.tok2id.get(tok, vocab.unk_id)\n        if not t.legal_mask()[i]:\n            r = t.remaining()\n            count_rule = (table.kind[i] == K_ATOM and t.pending is None and r is not None\n                          and r[table.elem_safe[i]] <= 0)\n            bad.append((k, "count" if count_rule else "grammar", tok))\n        t = t.push(i)\n        if limit is not None and t.pending is None and t.unsat() > limit + 1e-9:\n            bad.append((k, "dbe", tok))\n    if not t.eos_legal():\n        bad.append((len(tokens), "eos", None))\n    if max_len is not None and len(tokens) > max_len:\n        bad.append((len(tokens), "length", None))\n    return bad\n'
OFFICIAL_WORKER_SRC = '"""P0.1a official DreaMS leg: runs INSIDE the DreaMS venv (Python 3.11, upstream pins), never in the kernel.\n\nLoads the embedding model the way `casmi_dreams_colab.ipynb` did (`PreTrainedModel.from_name`), then\n  1. dumps the backbone args, the head hparams and the checkpoint SHA1s (A2, A3),\n  2. converts it once to `dreams_embed.pt` with build/dreams_infer.py `save_dreams` (A2),\n  3. embeds the parity spectra with the official `dreams_embeddings` over an MSData HDF5 (leg a),\n     exactly as cell 8 built the cache,\n  4. reloads `dreams_embed.pt` into the vendored network, strict, and embeds the same spectra (leg b,\n     venv copy; the notebook recomputes leg b in its own interpreter).\n\nUsage (from the notebook):\n    <venv python> dreams_official_worker.py --infer-dir DIR --ckpt embedding_model.ckpt \\\n        --parity parity_in.npz --out-pt dreams_embed.pt --out-emb official.npz --out-info info.json \\\n        --commit <sha>\n"""\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\n\n\ndef sha1(path, chunk=1 << 20):\n    h = hashlib.sha1()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(chunk), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef jsonable(v):\n    try:\n        json.dumps(v)\n        return v\n    except TypeError:\n        return repr(v)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--infer-dir", required=True)\n    ap.add_argument("--ckpt", default="", help="embedding_model.ckpt cached on Drive (\'\' = fetch from HF)")\n    ap.add_argument("--parity", required=True)\n    ap.add_argument("--out-pt", required=True)\n    ap.add_argument("--out-emb", required=True)\n    ap.add_argument("--out-info", required=True)\n    ap.add_argument("--commit", required=True)\n    ap.add_argument("--batch-size", type=int, default=64)\n    a = ap.parse_args()\n\n    sys.path.insert(0, a.infer_dir)\n    import numpy as np\n    import pandas as pd\n    import torch\n    import dreams_infer as di\n    import dreams.utils.data as du\n    from dreams.api import PreTrainedModel, dreams_embeddings\n    from dreams.definitions import DREAMS_EMBEDDING, PRETRAINED\n\n    PRETRAINED.mkdir(parents=True, exist_ok=True)\n    tgt = PRETRAINED / "embedding_model.ckpt"\n    if not tgt.exists() and a.ckpt:\n        shutil.copy(a.ckpt, tgt)       # the copy that built the cache; otherwise from_name fetches HF\'s\n    pm = PreTrainedModel.from_name(DREAMS_EMBEDDING)     # fetches ssl_model.ckpt from HF (A1)\n    model = pm.model\n    args = model.backbone.hparams["args"]\n    args_d = di.args_to_dict(args)\n    info = {\n        "dreams_commit": a.commit,\n        "python": sys.version.split()[0], "torch": torch.__version__,\n        "embedding_model_sha1": sha1(tgt), "ssl_model_sha1": sha1(PRETRAINED / "ssl_model.ckpt"),\n        "n_highest_peaks": pm.n_highest_peaks,\n        "dtype": str(next(model.parameters()).dtype),\n        "head_class": type(model).__name__,\n        "head_hparams": {k: jsonable(v) for k, v in dict(model.hparams).items()},\n        "backbone_args": args_d,\n        "backbone_args_all": {k: jsonable(v) for k, v in vars(args).items()},\n        "n_params": int(sum(p.numel() for p in model.parameters())),\n    }\n    assert args_d["charge_feature"] is False or args_d["charge_feature"] == 0, "charge_feature is set"\n    assert pm.n_highest_peaks == di.N_PEAKS, pm.n_highest_peaks\n    assert info["dtype"] == "torch.float32", info["dtype"]\n\n    meta = {k: info[k] for k in ("dreams_commit", "embedding_model_sha1", "ssl_model_sha1", "torch")}\n    di.save_dreams(a.out_pt, args, model.state_dict(), meta)\n    info["dreams_embed_sha1"] = sha1(a.out_pt)\n    info["state_dict_keys"] = len(di.inference_state_dict(model.state_dict()))\n\n    z = np.load(a.parity)\n    off, mz, it, prec = z["off"], z["mz"], z["it"], z["prec"]\n    pls = [np.stack([mz[off[i]:off[i + 1]], it[off[i]:off[i + 1]]]).astype(np.float32)\n           for i in range(len(prec))]\n\n    # leg (a): the official path, as casmi_dreams_colab.ipynb cell 8\n    t0 = time.time()\n    h5 = Path(a.out_emb).with_suffix(".hdf5")\n    h5.unlink(missing_ok=True)\n    df = pd.DataFrame({"spectrum": pls, "precursor_mz": prec.astype(np.float64)})\n    msdata = du.MSData.from_pandas(df, n_highest_peaks=di.N_PEAKS, hdf5_pth=h5, in_mem=False, mode="r")\n    emb_a = np.asarray(dreams_embeddings(msdata, batch_size=a.batch_size, progress_bar=False), np.float32)\n    info["t_official_s"] = time.time() - t0\n\n    # leg (b, venv copy): the converted file in the vendored network, strict load\n    t0 = time.time()\n    vend, vmeta = di.load_dreams(a.out_pt, device="cpu", expect_sha1=info["dreams_embed_sha1"])\n    emb_b = di.dreams_embed(vend, pls, prec, batch_size=a.batch_size)\n    info["t_vendored_venv_s"] = time.time() - t0\n    info["official_device"] = str(next(model.parameters()).device)\n\n    np.savez(a.out_emb, emb_a=emb_a, emb_b_venv=emb_b)\n    Path(a.out_info).write_text(json.dumps(info, indent=1, sort_keys=True))\n    print(json.dumps({k: info[k] for k in ("dreams_embed_sha1", "embedding_model_sha1", "ssl_model_sha1",\n                                           "n_highest_peaks", "head_class", "t_official_s")}))\n\n\nif __name__ == "__main__":\n    main()\n'

SRC_SHA1 = {}
for fname, text in [("dreams_infer.py", DREAMS_INFER_SRC), ("novel_decoder.py", NOVEL_DECODER_SRC),
                    ("dreams_official_worker.py", OFFICIAL_WORKER_SRC)]:
    (SRC / fname).write_text(text, encoding="utf-8")
    SRC_SHA1[fname] = sha1_text(text)
sys.path.insert(0, str(SRC))
import dreams_infer as di
import novel_decoder as nd
nd.check_constraints()
GATE_RESULT["src_sha1"] = SRC_SHA1
print(SRC_SHA1)

## P0.2 — cache audit

`emb_*.npy` chunks of 200,000 rows, aligned with `corpus_meta.parquet`: row counts, dimension,
finiteness.

In [ ]:
t0 = time.time()
META = pd.read_parquet(DOUT / "corpus_meta.parquet")
N = len(META)
n_chunk = (N + CHUNK - 1) // CHUNK
emb_files = sorted((DOUT / "emb").glob("emb_*.npy"))
assert [f.name for f in emb_files] == [f"emb_{i:04d}.npy" for i in range(n_chunk)], \
    f"expected {n_chunk} chunks, found {[f.name for f in emb_files]}"
EMB = [np.load(f, mmap_mode="r") for f in emb_files]
sizes = [e.shape[0] for e in EMB]
assert sum(sizes) == N and all(s == CHUNK for s in sizes[:-1]), sizes
assert all(e.shape[1] == 1024 and e.dtype == np.float32 for e in EMB)
finite = all(bool(np.isfinite(np.asarray(e)).all()) for e in EMB)
assert finite, "the cache holds non-finite values"

def cache_rows(rows):
    return np.stack([np.asarray(EMB[r // CHUNK][r % CHUNK]) for r in rows]).astype(np.float32)

GATE_RESULT["P0.2_cache"] = dict(rows=N, chunks=n_chunk, dim=1024, finite=finite,
                                 columns=list(META.columns), seconds=round(time.time() - t0))
save_gate()
print(GATE_RESULT["P0.2_cache"])

## P0.2 — replay cell 7 and attach collision energy

Cell 7 verbatim (adduct filter, `groupby("ingest_lib")` with per-library `sample(cap,
random_state=0)`, unusable-row drop), with `instrument_type` added and the peak arrays streamed
instead of held in memory. Sampling only depends on group sizes, so dropping the array columns
from the frame does not change which rows are picked. `_row` keeps each row's `train.parquet`
index; `collision_energy_ev` is read separately from its list offsets.

In [ ]:
t0 = time.time()
COLS = ["ingest_lib", "inchikey14", "normalized_smiles", "molecular_formula",
        "ionization_mode", "adduct", "precursor_mz", "instrument_type"]
tr = pq.read_table(TRAIN, columns=COLS).to_pandas()
N_TRAIN = len(tr)
tr["_row"] = np.arange(N_TRAIN)

tr = tr[tr.adduct.isin(TEST_ADDUCTS)]
parts = []
for lib, g in tr.groupby("ingest_lib"):
    cap = CAP_ENVEDA180 if lib == "enveda-180" else CAP_PER_LIB
    if cap and len(g) > cap:
        g = g.sample(cap, random_state=0)
    parts.append(g)
tr = pd.concat(parts).reset_index(drop=True)
print(f"after filter + caps: {len(tr):,} rows ({time.time() - t0:.0f}s)")

# pass 1 over the peak arrays: which selected rows cell 7 dropped as unusable, and peaks kept
pos = np.full(N_TRAIN, -1, np.int64)
pos[tr._row.to_numpy()] = np.arange(len(tr))
usable = np.zeros(len(tr), bool)
npk = np.zeros(len(tr), np.int16)
prec_sel = tr.precursor_mz.to_numpy(np.float64)

def list_parts(col):
    return col.offsets.to_numpy(), col.values.to_numpy(zero_copy_only=False)

base = 0
for batch in pq.ParquetFile(TRAIN).iter_batches(batch_size=100_000,
                                                 columns=["ms2_mzs", "ms2_normalized_intensities"]):
    n = batch.num_rows
    hit = np.flatnonzero(pos[base:base + n] >= 0)
    if len(hit):
        mo, mv = list_parts(batch.column(0))
        io_, iv = list_parts(batch.column(1))
        for local in hit:
            j = pos[base + local]
            p = di.dreams_peaks(mv[mo[local]:mo[local + 1]], iv[io_[local]:io_[local + 1]], prec_sel[j])
            if p is not None:
                usable[j] = True
                npk[j] = p.shape[1]
    base += n
assert base == N_TRAIN
tr = tr[usable].reset_index(drop=True)
npk = npk[usable]
tr["oob_mass"] = (tr.precursor_mz > 1000).astype(np.int8)
print(f"usable: {len(tr):,} ({(~usable).sum():,} dropped) ({time.time() - t0:.0f}s)")

# row-by-row comparison with the cached corpus_meta
def as_str(s):
    return s.astype(object).where(s.notna(), "<NULL>").astype(str).to_numpy()

cmp = {}
if len(tr) == N:
    for c in ["ingest_lib", "inchikey14", "adduct", "normalized_smiles", "molecular_formula",
              "ionization_mode"]:
        cmp[c] = int((as_str(tr[c]) != as_str(META[c])).sum())
    cmp["precursor_mz"] = int((tr.precursor_mz.to_numpy(np.float64) != META.precursor_mz.to_numpy(np.float64)).sum())
    cmp["oob_mass"] = int((tr.oob_mass.to_numpy() != META.oob_mass.to_numpy()).sum())
REPLAY_OK = len(tr) == N and all(cmp[c] == 0 for c in ["ingest_lib", "inchikey14", "adduct", "precursor_mz"])
print(f"replay rows {len(tr):,} vs cache {N:,}; mismatches per column {cmp}")
print(f"G0.2 replay: {'MATCH' if REPLAY_OK else 'MISMATCH -> HOLD (decision rule 2)'}")

# collision energy per corpus row, from the list column's offsets
ce_col = pq.read_table(TRAIN, columns=["collision_energy_ev"]).column(0).combine_chunks()
if not (pa.types.is_list(ce_col.type) or pa.types.is_large_list(ce_col.type)):
    raise TypeError(f"collision_energy_ev is {ce_col.type}, expected a list")
def gather_lists(offs, vals, rows):
    """The sub-lists of `rows` from a flattened list column -> (new offsets, values), vectorised."""
    lens = (offs[rows + 1] - offs[rows]).astype(np.int64)
    new_offs = np.concatenate([[0], np.cumsum(lens)])
    idx = np.repeat(offs[rows] - new_offs[:-1], lens) + np.arange(new_offs[-1])
    return new_offs, vals[idx]

def list_means(new_offs, v):
    """Per list: (number of finite positive values, their mean or NaN)."""
    n = len(new_offs) - 1
    seg = np.repeat(np.arange(n), np.diff(new_offs))
    fin = np.isfinite(v) & (v > 0)
    n_fin = np.bincount(seg[fin], minlength=n)
    mean = np.full(n, np.nan)
    ok = n_fin > 0
    mean[ok] = np.bincount(seg[fin], weights=v[fin], minlength=n)[ok] / n_fin[ok]
    return n_fin, mean

offs, vals = list_parts(ce_col)
rows = tr._row.to_numpy()
new_offs, v = gather_lists(offs, vals.astype(np.float64), rows)
n_fin, ce_mean = list_means(new_offs, v)
ok_ce = n_fin > 0
del ce_col, vals
CE = pa.table({"row": pa.array(np.arange(len(rows))), "train_row": pa.array(rows),
               "ce_list": pa.ListArray.from_arrays(pa.array(new_offs.astype(np.int32)), pa.array(v)),
               "n_ce": pa.array(n_fin.astype(np.int16)), "ce_mean": pa.array(ce_mean),
               "instrument_type": pa.array(tr.instrument_type.astype(object).to_numpy())})
if REPLAY_OK:
    pq.write_table(CE, GATE / "corpus_ce.parquet")

q = np.nanquantile(ce_mean, [0.2, 0.4, 0.6, 0.8]) if ok_ce.any() else []
CE_BINS = sorted(set(float(round(x)) for x in q))
hist_edges = [0, 10, 20, 30, 40, 50, 60, 80, 100, 1e9]
ce_hist = np.histogram(ce_mean[ok_ce], bins=hist_edges)[0]
GATE_RESULT["P0.2"] = dict(
    replay_rows=len(tr), cache_rows=N, mismatches=cmp, replay_match=REPLAY_OK,
    corpus_ce_written=REPLAY_OK, rows=len(tr), unique_ik14=int(tr.inchikey14.nunique()),
    per_adduct=tr.adduct.value_counts().to_dict(), per_mode=tr.ionization_mode.value_counts().to_dict(),
    per_lib=tr.ingest_lib.value_counts().to_dict(), oob_share=float(tr.oob_mass.mean()),
    ce_missing_share=float((~ok_ce).mean()), ce_multi_share=float((n_fin > 1).mean()),
    ce_hist=dict(zip([f"{a}-{b}" for a, b in zip(hist_edges[:-1], hist_edges[1:])], ce_hist.tolist())),
    ce_bins_proposed=CE_BINS, ce_bins_note="quantile edges of ce_mean (20/40/60/80 %) + an 'unknown' bin",
    instrument_top=tr.instrument_type.astype(str).value_counts().head(10).to_dict(),
    seconds=round(time.time() - t0))
save_gate()
print(json.dumps({k: GATE_RESULT["P0.2"][k] for k in ("rows", "unique_ik14", "oob_share", "ce_missing_share",
                                                      "ce_multi_share", "ce_bins_proposed", "ce_hist")}, indent=1))

## P0.1a — parity spectra

200 corpus rows stratified by adduct × (precursor > 1000 Da) × peaks kept (2–9 / 10–39 / 40–100),
spread evenly across strata. Their peak lists are rebuilt from `train.parquet` with `dreams_peaks`
(= cell 7 `peak_list`, tested offline).

In [ ]:
def stratified_pick(S, keys, n, seed):
    """n rows spread evenly over the strata: round-robin by a random rank within each stratum."""
    S = S.assign(r=np.random.default_rng(seed).random(len(S)))
    S["rank"] = S.sort_values("r").groupby(keys).cumcount()
    return S.sort_values(["rank", "r"]).head(n).sort_values("row").reset_index(drop=True)

t0 = time.time()
S = pd.DataFrame({"row": np.arange(len(tr)), "adduct": tr.adduct.to_numpy(), "oob": tr.oob_mass.to_numpy(),
                  "npk_bin": np.digitize(npk, [10, 40])})
PAR = stratified_pick(S, ["adduct", "oob", "npk_bin"], N_PARITY, SEED)
par_rows = PAR.row.to_numpy()
par_train = tr._row.to_numpy()[par_rows]
par_prec = tr.precursor_mz.to_numpy(np.float64)[par_rows]
print(f"{len(PAR)} parity rows over {S.groupby(['adduct', 'oob', 'npk_bin']).ngroups} strata")

want = {int(r): i for i, r in enumerate(par_train)}
PLS = [None] * len(PAR)
base = 0
for batch in pq.ParquetFile(TRAIN).iter_batches(batch_size=100_000,
                                                 columns=["ms2_mzs", "ms2_normalized_intensities"]):
    n = batch.num_rows
    hits = [(r - base, want[r]) for r in range(base, base + n) if r in want]
    if hits:
        mo, mv = list_parts(batch.column(0))
        io_, iv = list_parts(batch.column(1))
        for local, j in hits:
            PLS[j] = di.dreams_peaks(mv[mo[local]:mo[local + 1]], iv[io_[local]:io_[local + 1]], par_prec[j])
    base += n
assert all(p is not None for p in PLS), "a parity row lost its peaks"
lens = np.array([p.shape[1] for p in PLS])
par_off = np.concatenate([[0], np.cumsum(lens)])
np.savez(WORK / "parity_in.npz", off=par_off, mz=np.concatenate([p[0] for p in PLS]),
         it=np.concatenate([p[1] for p in PLS]), prec=par_prec, row=par_rows)
EMB_C = cache_rows(par_rows)
print(f"peaks fetched ({time.time() - t0:.0f}s); cache vectors {EMB_C.shape}")

## P0.1a — official DreaMS in its own venv: conversion + leg (a)

Clones DreaMS at the vendored commit, installs it with its own pins into a Python 3.11 venv, and
runs `dreams_official_worker.py`: `PreTrainedModel.from_name`, the args / hparams dump, the
one-off conversion to `dreams_embed.pt`, and the official `dreams_embeddings` on the parity rows.
`embedding_model.ckpt` is the Drive copy that built the cache; `ssl_model.ckpt` comes from HF.

In [ ]:
DREAMS_DIR = Path("/content/DreaMS")
DVENV = Path("/content/dreams_venv")
DPY = DVENV / "bin" / "python"
VENV_ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}
t0 = time.time()
if not DPY.exists():
    sh(f"{sys.executable} -m pip install -q uv")
    if not DREAMS_DIR.exists():
        sh(f"git clone -q https://github.com/pluskal-lab/DreaMS.git {DREAMS_DIR}")
    sh(f"cd {DREAMS_DIR} && git checkout -q {DREAMS_COMMIT}")
    sh(f"uv venv -q --python 3.11 {DVENV}", env=VENV_ENV, timeout=1800)
    sh(f"uv pip install -q --python {DPY} -e {DREAMS_DIR}", env=VENV_ENV, timeout=3600)
head = sh(f"cd {DREAMS_DIR} && git rev-parse HEAD").strip()
assert head == DREAMS_COMMIT, head
t_install = time.time() - t0

ckpt = DOUT / "embedding_model.ckpt"
sh(f"{DPY} {SRC / 'dreams_official_worker.py'} --infer-dir {SRC} "
   f"--ckpt {ckpt if ckpt.exists() else chr(34) * 2} --parity {WORK / 'parity_in.npz'} "
   f"--out-pt {WORK / 'dreams_embed.pt'} --out-emb {WORK / 'official.npz'} "
   f"--out-info {WORK / 'dreams_info.json'} --commit {DREAMS_COMMIT}", env=VENV_ENV, timeout=3600)
DINFO = json.loads((WORK / "dreams_info.json").read_text())
OFF = np.load(WORK / "official.npz")
EMB_A, EMB_B_VENV = OFF["emb_a"], OFF["emb_b_venv"]
print(json.dumps({k: DINFO[k] for k in ("embedding_model_sha1", "ssl_model_sha1", "dreams_embed_sha1",
                                        "n_highest_peaks", "head_class", "head_hparams", "torch")}, indent=1))

## P0.1a — leg (b) in this kernel, and the three-way comparison

In [ ]:
t0 = time.time()
VEND, VMETA = di.load_dreams(WORK / "dreams_embed.pt", device="cpu", expect_sha1=DINFO["dreams_embed_sha1"])
EMB_B = di.dreams_embed(VEND, PLS, par_prec, batch_size=64)
t_b = time.time() - t0

def rowcos(x, y):
    x = x.astype(np.float64); y = y.astype(np.float64)
    return (x * y).sum(1) / (np.linalg.norm(x, axis=1) * np.linalg.norm(y, axis=1))

cos_ab, cos_bc, cos_bb = rowcos(EMB_A, EMB_B), rowcos(EMB_B, EMB_C), rowcos(EMB_B, EMB_B_VENV)
cos_ac = rowcos(EMB_A, EMB_C)
LEG_A = bool(cos_ab.min() >= COS_OFFICIAL)
LEG_C = bool(cos_bc.min() >= COS_CACHE)
worst = PAR.assign(cos_bc=cos_bc, cos_ab=cos_ab, prec=par_prec).sort_values("cos_bc").head(5)
print(f"(a) official vs (b) vendored: min {cos_ab.min():.6f}  (>= {COS_OFFICIAL}: {LEG_A})")
print(f"(b) vendored vs (c) cache:    min {cos_bc.min():.6f}  median {np.median(cos_bc):.6f}  (>= {COS_CACHE}: {LEG_C})")
print(f"(a) official vs (c) cache:    min {cos_ac.min():.6f}")
print(f"(b) kernel vs (b) venv:       min {cos_bb.min():.6f}")
print(worst.to_string())

np.savez(GATE / "parity_200.npz", row=par_rows, train_row=par_train, prec=par_prec, off=par_off,
         mz=np.concatenate([p[0] for p in PLS]), it=np.concatenate([p[1] for p in PLS]),
         adduct=PAR.adduct.to_numpy().astype(str), emb_a=EMB_A, emb_b=EMB_B, emb_c=EMB_C)
shutil.copy2(WORK / "dreams_embed.pt", GATE / "dreams_embed.pt")
GATE_RESULT["P0.1a"] = dict(
    dreams_info={k: v for k, v in DINFO.items() if k != "backbone_args_all"},
    fine_tune_note="ContrastiveHead hparams above; provisional source MoNA [M+H]+, Murcko split "
                   "(dreams/training/fine_tune.sh:45-71) -- the checkpoint does not name its data",
    venv_install_s=round(t_install), t_vendored_cpu_per_spectrum_ms=1000 * t_b / len(PLS),
    cos_ab_min=float(cos_ab.min()), cos_bc_min=float(cos_bc.min()), cos_bc_median=float(np.median(cos_bc)),
    cos_ac_min=float(cos_ac.min()), cos_bb_venv_min=float(cos_bb.min()),
    leg_official=LEG_A, leg_cache=LEG_C, n_parity=len(PLS),
    strata=int(S.groupby(["adduct", "oob", "npk_bin"]).ngroups),
    worst_cache_rows=worst.drop(columns=["r"]).to_dict("records"))
save_gate()

## P0.3 — SELFIES round-trip, `MAX_LEN`, vocabulary

Unique corpus structures by ik14 → `prep_structure` (stereo + isotopes stripped, single component,
C/H/N/O/P/S/F/Cl/Br/I) → `sf.encoder` → `sf.decoder` → ik14 with this RDKit. The round-trip rate is
over `ok` + `encode` structures (single-component, allowed elements); `parse`, `multi` and
`element` are the reported exclusions.

In [ ]:
t0 = time.time()
U = META.drop_duplicates("inchikey14")[["inchikey14", "normalized_smiles"]].reset_index(drop=True)
CHARGED = re.compile(r"\[[^\]]*[+-]\d*\]")

def prep_one(smi):
    st, can, s, f = nd.prep_structure(smi)
    rt = k_can = None
    if st == "ok":
        k_can = nd.ik14(can)
        k_dec = nd.ik14(sf.decoder(s))
        rt = k_dec is not None and k_dec == k_can
    return st, can, s, f, rt, k_can

with pool() as p:
    R = p.map(prep_one, U.normalized_smiles.tolist(), chunksize=256)
U["status"] = [r[0] for r in R]; U["can"] = [r[1] for r in R]; U["selfies"] = [r[2] for r in R]
U["formula"] = [r[3] for r in R]; U["rt"] = [r[4] for r in R]; U["k_can"] = [r[5] for r in R]
U.loc[(U.status == "ok") & ~U.rt.astype(bool), "status"] = "roundtrip"
denom = U.status.isin(["ok", "roundtrip", "encode"])
RT_RATE = float((U.status == "ok").sum() / denom.sum())
status_counts = U.status.value_counts().to_dict()
examples = {st: U[U.status == st].normalized_smiles.head(8).tolist() for st in status_counts if st != "ok"}
label_agree = float((U.k_can[U.status == "ok"] == U.inchikey14[U.status == "ok"]).mean())

OK = U[U.status == "ok"].reset_index(drop=True)
OK["toks"] = [list(sf.split_selfies(s)) for s in OK.selfies]
L = OK.toks.map(len).to_numpy()
MAX_LEN = int(np.ceil(np.quantile(L, MAXLEN_Q)))
above = float((L > MAX_LEN).mean())
charged_ok = float(OK.can.map(lambda s: bool(CHARGED.search(s))).mean())

alphabet = sf.get_alphabet_from_selfies(OK.selfies.tolist()) | sf.get_semantic_robust_alphabet()
VOCAB = nd.SelfiesVocab.from_alphabet(alphabet)
VOCAB.to_json(GATE / "selfies_vocab.json")
vocab_sha1 = di.sha1(GATE / "selfies_vocab.json")
corpus_only = sorted(sf.get_alphabet_from_selfies(OK.selfies.tolist()) - sf.get_semantic_robust_alphabet())

G03 = RT_RATE >= RT_MIN
GATE_RESULT["P0.3"] = dict(
    unique_ik14=len(U), status=status_counts, examples=examples,
    roundtrip_rate=RT_RATE, roundtrip_denominator=int(denom.sum()),
    excluded_share=float(1 - denom.mean()), label_ik14_agreement=label_agree,
    charged_share_of_ok=charged_ok, max_len=MAX_LEN, max_len_quantile=MAXLEN_Q, share_above_max_len=above,
    len_p50=float(np.median(L)), len_max=int(L.max()), vocab_size=len(VOCAB), vocab_sha1=vocab_sha1,
    corpus_tokens_outside_robust_alphabet=corpus_only, passed=G03, seconds=round(time.time() - t0))
save_gate()
print(f"round-trip {RT_RATE:.4f} over {int(denom.sum()):,} (>= {RT_MIN}: {G03}); status {status_counts}")
print(f"MAX_LEN {MAX_LEN} (p{MAXLEN_Q * 100:g}), {above:.4%} above; vocab {len(VOCAB)} tokens, "
      f"{len(corpus_only)} outside the robust alphabet; label ik14 agreement {label_agree:.4f}")

## P0.4 — tracker parity and mask soundness

**Parity:** 10k random round-trip-OK structures, every prefix: fast `Tracker` vs `reference_state`
(`sf.decoder(prefix)` + RDKit) on element counts and rings + π bonds.
**Soundness:** every round-trip-OK structure walked under its own formula (`walk_truth`, DBE prune
on, so its violations are counted too). Count, `</s>` and grammar violations mean the mask would
cut off a true structure. Length: structures above `MAX_LEN` are out of scope by construction
(reported in P0.3); the walk confirms that is the only length violation.

In [ ]:
t0 = time.time()
TT = nd.TokenTable(VOCAB)
rng = np.random.default_rng(SEED + 1)
par_ix = rng.choice(len(OK), size=min(N_TRACKER_PARITY, len(OK)), replace=False)

def parity_one(i):
    return nd.tracker_parity(TT, OK.toks[i])

with pool() as p:
    PB = p.map(parity_one, par_ix.tolist(), chunksize=16)
n_prefix = int(L[par_ix].sum())
par_bad = [(int(i), b[:3]) for i, b in zip(par_ix, PB) if b]
print(f"parity: {len(par_bad)} / {len(par_ix)} structures with a mismatch over {n_prefix:,} prefixes "
      f"({time.time() - t0:.0f}s)")

def walk_one(i):
    return nd.walk_truth(TT, OK.toks[i], OK.formula[i], max_len=MAX_LEN, dbe_prune=True)

t1 = time.time()
with pool() as p:
    WB = p.map(walk_one, range(len(OK)), chunksize=64)
t_walk = time.time() - t1
rules = ["count", "grammar", "eos", "length", "dbe"]
by_rule = {r: int(sum(any(v[1] == r for v in b) for b in WB)) for r in rules}
ex_rule = {r: [(OK.can[i], [list(map(str, v)) for v in b if v[1] == r][:3])
               for i, b in enumerate(WB) if any(v[1] == r for v in b)][:6] for r in rules}
prune_applies = int(sum(nd.dbe_prune_applies(f) for f in OK.formula))
DBE_PRUNE = by_rule["dbe"] == 0
dbe_charged = [OK.can[i] for i, b in enumerate(WB) if any(v[1] == "dbe" for v in b)]
dbe_charged_share = float(np.mean([bool(CHARGED.search(s)) for s in dbe_charged])) if dbe_charged else 0.0
length_consistent = by_rule["length"] == int((L > MAX_LEN).sum())

# fast-tracker cost per decoding step (single process) -- an input to P4's search budget
tt0, steps = time.time(), 0
for i in range(min(1000, len(OK))):
    t = nd.Tracker(TT, OK.formula[i])
    for tok in OK.toks[i]:
        t.legal_mask(); t = t.push(VOCAB.tok2id[tok]); steps += 1
mask_us = 1e6 * (time.time() - tt0) / max(steps, 1)

G04 = not par_bad and by_rule["count"] == 0 and by_rule["eos"] == 0 and length_consistent
GATE_RESULT["P0.4"] = dict(
    parity_structures=len(par_ix), parity_prefixes=n_prefix, parity_mismatch_structures=len(par_bad),
    parity_examples=par_bad[:5], walk_structures=len(OK), violations_by_rule=by_rule,
    violation_examples=ex_rule, grammar_note="a grammar violation means a true token is masked for a "
    "non-count reason; not a pre-registered rule, reviewed before P1 if non-zero",
    dbe_prune_applies_to=prune_applies, dbe_prune=DBE_PRUNE, dbe_violation_charged_share=dbe_charged_share,
    length_violations_equal_share_above_max_len=length_consistent,
    mask_us_per_step=mask_us, walk_seconds=round(t_walk), passed=G04, seconds=round(time.time() - t0))
save_gate()
print(f"walk over {len(OK):,} structures ({t_walk:.0f}s): violations by rule {by_rule}")
print(f"DBE prune applies to {prune_applies:,}; DBE_PRUNE = {DBE_PRUNE} "
      f"(violators charged: {dbe_charged_share:.0%})")
print(f"fast tracker: {mask_us:.0f} us per step (legal_mask + push)")
for r in rules:
    if ex_rule[r]:
        print(f"  {r}: {ex_rule[r][:3]}")

## P0.5 — MassSpecGym molecules (`msg_ik14.npy`) and the P1 preview

Built before massspecgym is installed, so the install cannot touch the RDKit used for ik14.

In [ ]:
t0 = time.time()
from huggingface_hub import hf_hub_download
tsv = hf_hub_download(repo_id="roman-bushuiev/MassSpecGym", filename="data/MassSpecGym1.5.tsv",
                      repo_type="dataset")
MSG = pd.read_csv(tsv, sep="\t", usecols=lambda c: c in ("smiles", "inchikey", "fold"))
msg_smiles = MSG.smiles.dropna().unique().tolist()
with pool() as p:
    msg_k = p.map(nd.ik14, msg_smiles, chunksize=256)
MSG_IK14 = np.array(sorted({k for k in msg_k if k}), dtype="U14")
np.save(GATE / "msg_ik14.npy", MSG_IK14)
msg_label_agree = None
if "inchikey" in MSG.columns:
    lab = dict(zip(MSG.smiles, MSG.inchikey.astype(str).str[:14]))
    msg_label_agree = float(np.mean([lab.get(s) == k for s, k in zip(msg_smiles, msg_k)]))
msg_set = set(MSG_IK14.tolist())
lib_u = META.drop_duplicates(["ingest_lib", "inchikey14"])
preview = (lib_u.assign(outside=~lib_u.inchikey14.isin(msg_set)).groupby("ingest_lib")
                .agg(unique_ik14=("inchikey14", "size"), outside_msg=("outside", "sum")))
preview["share_outside"] = preview.outside_msg / preview.unique_ik14
u_all = META.inchikey14.drop_duplicates()
P1_PREVIEW = dict(corpus_unique=int(len(u_all)), outside_msg=int((~u_all.isin(msg_set)).sum()),
                  per_lib=preview.reset_index().to_dict("records"))
print(f"MassSpecGym: {len(msg_smiles):,} SMILES -> {len(MSG_IK14):,} ik14 "
      f"(label agreement {msg_label_agree}) ({time.time() - t0:.0f}s)")
print(f"corpus ik14 outside MassSpecGym: {P1_PREVIEW['outside_msg']:,} / {P1_PREVIEW['corpus_unique']:,}")
print(preview.sort_values("unique_ik14", ascending=False).to_string())

## P0.5 — massspecgym install and imports

`--no-deps -e` at the pinned commit, then the import closure (A5) with the kernel's numpy / torch /
RDKit / pandas held by a constraints file. The imports and the `SelfiesTokenizer` id check run in
a fresh interpreter, so modules already loaded here cannot mask a failure.

In [ ]:
t0 = time.time()
MSGYM_DIR = Path("/content/MassSpecGym")
if not MSGYM_DIR.exists():
    sh(f"git clone -q https://github.com/pluskal-lab/MassSpecGym.git {MSGYM_DIR}")
sh(f"cd {MSGYM_DIR} && git checkout -q {MSGYM_COMMIT}")
sh(f"{sys.executable} -m pip install -q --no-deps -e {MSGYM_DIR}")
held = {m: kernel_version(m) for m in ("numpy", "torch", "rdkit", "pandas", "selfies")}
pins = {**{m: v for m, v in held.items() if v}, "rdkit": "2026.3.3"}      # PyPI spells it 2026.3.3
(WORK / "constraints.txt").write_text("".join(f"{m}=={v}\n" for m, v in pins.items()))
DEPS = ("pytorch-lightning torchmetrics torch_geometric myopic-mces pulp matchms chemparse chemformula "
        "tokenizers huggingface_hub seaborn joblib pyyaml")
dep_log = sh(f"{sys.executable} -m pip install -q {DEPS} -c {WORK / 'constraints.txt'}", check=False)
after = {m: kernel_version(m) for m in held}

IMPORT_CHECK = r'''
import importlib, json
out = {}
try:
    import distutils.util; out["distutils"] = "ok"
except Exception as e:
    out["distutils"] = f"{type(e).__name__}: {e}"
try:
    import pulp; out["pulp_solvers"] = pulp.listSolvers(onlyAvailable=True)
except Exception as e:
    out["pulp_solvers"] = f"{type(e).__name__}: {e}"
for mod, name in [("massspecgym.models.tokenizers", "SelfiesTokenizer"),
                  ("massspecgym.data.transforms", "MolToFormulaVector"),
                  ("massspecgym.models.de_novo.base", "DeNovoMassSpecGymModel"),
                  ("massspecgym.models.de_novo.random", "RandomDeNovo")]:
    try:
        getattr(importlib.import_module(mod), name); out[name] = "ok"
    except Exception as e:
        out[name] = f"{type(e).__name__}: {e}"[:500]
print("JSON>>" + json.dumps(out))
'''

def run_check(src, *args):
    (WORK / "check.py").write_text(src)
    r = subprocess.run([sys.executable, str(WORK / "check.py"), *map(str, args)], capture_output=True, text=True)
    line = [ln for ln in r.stdout.splitlines() if ln.startswith("JSON>>")]
    if not line:
        print(r.stdout[-2000:], r.stderr[-4000:])
        return {"error": r.stderr[-1000:]}
    return json.loads(line[-1][len("JSON>>"):])

IMP = run_check(IMPORT_CHECK)
if IMP.get("distutils") != "ok":
    # A5: misc_utils.py:3 imports distutils (gone in 3.12 without setuptools' shim) -- patch the clone
    mu = MSGYM_DIR / "massspecgym" / "simulation_utils" / "misc_utils.py"
    mu.write_text(mu.read_text().replace(
        "from distutils.util import strtobool",
        "def strtobool(v):\n    return {'y': 1, 'yes': 1, 't': 1, 'true': 1, 'on': 1, '1': 1}.get(str(v).lower(), 0)"))
    IMP = run_check(IMPORT_CHECK)
    IMP["distutils_patched"] = True
print(json.dumps(IMP, indent=1))

TOK_CHECK = r'''
import json, sys
import selfies as sf
from massspecgym.models.tokenizers import SelfiesTokenizer
vocab = json.load(open(sys.argv[1])); smiles = json.load(open(sys.argv[2])); max_len = int(sys.argv[3])
tok = SelfiesTokenizer(selfies_train=[t for t in vocab if t.startswith("[")], max_len=max_len + 2)
tv = tok.get_vocab()
bad, n = [], 0
for s in smiles:
    toks = list(sf.split_selfies(sf.encoder(s, strict=False)))
    if len(toks) > max_len:
        continue
    ours = [vocab["<s>"]] + [vocab.get(t, vocab["<unk>"]) for t in toks] + [vocab["</s>"]]
    ours += [vocab["<pad>"]] * (max_len + 2 - len(ours))
    n += 1
    if tok.encode(s).ids != ours:
        bad.append(s)
print("JSON>>" + json.dumps(dict(same_vocab=tv == vocab, n_tokenizer=len(tv), n_ours=len(vocab),
                                 checked=n, mismatches=len(bad), examples=bad[:3])))
'''
tok_smiles = OK.can.sample(min(1000, len(OK)), random_state=SEED).tolist()
(WORK / "tok_smiles.json").write_text(json.dumps(tok_smiles))
TOK = (run_check(TOK_CHECK, GATE / "selfies_vocab.json", WORK / "tok_smiles.json", MAX_LEN)
       if IMP.get("SelfiesTokenizer") == "ok" else {"skipped": "SelfiesTokenizer does not import"})
print(json.dumps(TOK, indent=1))

SYMBOLS = ("SelfiesTokenizer", "MolToFormulaVector", "DeNovoMassSpecGymModel", "RandomDeNovo")
G05 = all(IMP.get(s) == "ok" for s in SYMBOLS) and (GATE / "msg_ik14.npy").exists()
GATE_RESULT["P0.5"] = dict(
    imports=IMP, tokenizer_parity=TOK, versions_before=held, versions_after=after,
    versions_changed={m: (held[m], after[m]) for m in held if held[m] != after[m]},
    msg_smiles=len(msg_smiles), msg_ik14=len(MSG_IK14), msg_label_ik14_agreement=msg_label_agree,
    p1_preview=P1_PREVIEW, selfies_licence="Apache-2.0 (selfies " + SELFIES_PIN + ")",
    passed=G05, seconds=round(time.time() - t0))
save_gate()
print(f"G0.5 imports + msg_ik14: {G05}; versions changed by the install: {GATE_RESULT['P0.5']['versions_changed']}")

## Kaggle upload (`casmi26-novel-offline`)

`dreams_embed.pt`, `parity_200.npz`, the selfies wheel, the vocabulary, a 200-structure tracker
smoke set and the manifest the Kaggle notebook checks against.

In [ ]:
if UPLOAD.exists():
    shutil.rmtree(UPLOAD)
(UPLOAD / "wheels").mkdir(parents=True)
sh(f"{sys.executable} -m pip download -q --no-deps --only-binary=:all: selfies=={SELFIES_PIN} -d {UPLOAD / 'wheels'}")
for f in ("dreams_embed.pt", "parity_200.npz", "selfies_vocab.json"):
    shutil.copy2(GATE / f, UPLOAD / f)
rng = np.random.default_rng(SEED + 2)
smoke_ix = rng.choice(len(OK), size=min(N_SMOKE, len(OK)), replace=False)
(UPLOAD / "tracker_smoke.json").write_text(json.dumps(
    [dict(selfies=OK.selfies[i], formula=OK.formula[i]) for i in smoke_ix]))
MANIFEST = dict(
    dreams_embed_sha1=DINFO["dreams_embed_sha1"], dreams_commit=DREAMS_COMMIT,
    embedding_model_sha1=DINFO["embedding_model_sha1"], ssl_model_sha1=DINFO["ssl_model_sha1"],
    src_sha1={k: SRC_SHA1[k] for k in ("dreams_infer.py", "novel_decoder.py")},
    parity_sha1=di.sha1(UPLOAD / "parity_200.npz"), vocab_sha1=vocab_sha1,
    selfies=SELFIES_PIN, max_len=MAX_LEN, dbe_prune=DBE_PRUNE, cos_kaggle_min=COS_KAGGLE,
    colab=GATE_RESULT["env"])
(UPLOAD / "manifest.json").write_text(json.dumps(MANIFEST, indent=1))
(UPLOAD / "dataset-metadata.json").write_text(json.dumps(
    {"title": "casmi26-novel-offline", "id": f"{KAGGLE_USER}/casmi26-novel-offline",
     "licenses": [{"name": "other"}]}, indent=1))
size = sum(p.stat().st_size for p in UPLOAD.rglob("*") if p.is_file())
print(f"wrote {UPLOAD} ({size / 1e6:.0f} MB): {sorted(p.name for p in UPLOAD.rglob('*') if p.is_file())}")
GATE_RESULT["upload"] = dict(path=str(UPLOAD), mb=size / 1e6, manifest=MANIFEST)
save_gate()

## Gate report (Colab half)

In [ ]:
def verdict():
    items, holds, fails, notes = {}, [], [], []
    if LEG_A and LEG_C:
        items["G0.1"] = "PASS (Colab legs) -- Kaggle leg pending"
    elif LEG_A:
        items["G0.1"] = "HOLD: cache leg failed (decision rule 1: cache built by another DreaMS commit)"
        holds.append("G0.1")
    else:
        items["G0.1"] = f"FAIL: official vs vendored min cos {cos_ab.min():.6f} < {COS_OFFICIAL}"
        fails.append("G0.1")
    items["G0.2"] = "PASS" if REPLAY_OK else "HOLD: replay mismatch (decision rule 2)"
    if not REPLAY_OK:
        holds.append("G0.2")
    items["G0.3"] = "PASS" if G03 else f"FAIL: round-trip {RT_RATE:.4f} < {RT_MIN}"
    if not G03:
        fails.append("G0.3")
    if G04:
        items["G0.4"] = "PASS" + ("" if DBE_PRUNE else " (DBE_PRUNE disabled, decision rule 3)")
    else:
        items["G0.4"] = (f"FAIL: parity mismatches {len(par_bad)}, count {by_rule['count']}, "
                         f"eos {by_rule['eos']}, length consistent {length_consistent}")
        fails.append("G0.4")
    if by_rule["grammar"]:
        notes.append(f"{by_rule['grammar']} structures with grammar violations -- review before P1")
    items["G0.5"] = "PASS" if G05 else "FAIL: see P0.5 imports"
    if not G05:
        fails.append("G0.5")
    head = "FAIL" if fails else ("HOLD" if holds else "PASS (Colab half) -- G0.1 Kaggle leg pending")
    return items, head, notes

ITEMS, HEAD, NOTES = verdict()
GATE_RESULT.update(verdict=HEAD, items=ITEMS, notes=NOTES, finished=time.strftime("%Y-%m-%d %H:%M"))
save_gate()

print("=" * 88)
print(f"VERDICT: {HEAD}")
print("=" * 88)
for k, v in ITEMS.items():
    print(f"{k}  {v}")
for n in NOTES:
    print(f"NOTE  {n}")
print(f"\nG0.1  cos(a,b) min {cos_ab.min():.6f} | cos(b,c) min {cos_bc.min():.6f} | dreams_embed.pt {DINFO['dreams_embed_sha1']}")
print(f"G0.2  replay {len(tr):,} vs {N:,} rows; mismatches {cmp}; CE missing {GATE_RESULT['P0.2']['ce_missing_share']:.3f}; CE_BINS {CE_BINS}")
print(f"G0.3  round-trip {RT_RATE:.4f}; status {status_counts}; MAX_LEN {MAX_LEN}; vocab {len(VOCAB)} ({vocab_sha1})")
print(f"G0.4  parity {len(par_bad)} bad / {len(par_ix)}; violations {by_rule}; DBE_PRUNE {DBE_PRUNE}; {mask_us:.0f} us/step")
print(f"G0.5  imports {[IMP.get(s) for s in SYMBOLS]}; tokenizer parity {TOK}; msg ik14 {len(MSG_IK14):,}; "
      f"corpus outside MSG {P1_PREVIEW['outside_msg']:,}/{P1_PREVIEW['corpus_unique']:,}")
print(f"\nfull report: {GATE / 'phase0_colab_gate.json'}")
print(f"next: upload {UPLOAD} as casmi26-novel-offline and run casmi_novel_p0_kaggle.ipynb (T4 x2, internet off)")